# Notebook 06 — BraTS Data Preparation & 3D Patch Extraction

**Project:** NeuroSeg  
**Dataset:** BraTS 2024 Adult Glioma  
**Purpose:** Build a reusable data pipeline for our custom 3D U-Net, UNETR and Swin UNETR experiments.

## What this notebook does
1. Discover patient cases and verify the five NIfTI files per case.
2. Load the four MRI modalities and segmentation labels.
3. Check shape/affine alignment and preserve the BraTS 2024 labels (0–4).
4. Normalize each modality independently within its non-zero region.
5. Create a fixed patient-level train/validation/test split.
6. Implement tumour-aware 3D patch extraction.
7. Build and inspect a PyTorch Dataset/DataLoader.
8. Save the split manifest and reusable Python module.

**Important:** nnU-Net has its own self-configuring preprocessing pipeline. We will not force this custom normalization/patch pipeline onto nnU-Net. This notebook is intended for our custom 3D U-Net and Transformer-based implementations, with model-specific patch-size adjustments where needed.

**Data integrity:** No resampling is performed here. We verify that modalities and the segmentation have matching shapes and affines. If a case fails these checks, investigate it rather than silently resizing it.

## 1. Environment and configuration

Run this notebook in Kaggle with the BraTS dataset attached. The default path below matches the dataset location used in our earlier notebooks. If Kaggle mounts it at a different path, update `DATA_ROOT`.

In [ ]:
from pathlib import Path
import sys
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nibabel as nib
import torch

from torch.utils.data import DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DATA_ROOT = Path(
    "/kaggle/input/datasets/i212385nomanarif/2024-brats-glioma"
)

WORK_DIR = Path("/kaggle/working/neuroseg_data")
WORK_DIR.mkdir(parents=True, exist_ok=True)

print("Dataset path exists:", DATA_ROOT.exists())
print("Working directory:", WORK_DIR)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Discover patient cases

Each patient should contain:
- `-t1n.nii`: T1-native
- `-t1c.nii`: T1-contrast enhanced
- `-t2w.nii`: T2-weighted
- `-t2f.nii`: FLAIR
- `-seg.nii`: segmentation mask

The discovery function supports both `.nii` and `.nii.gz` files and searches recursively.

In [ ]:
# The helper module is created in Section 9. For now, discover files directly.
MODALITIES = ("t1n", "t1c", "t2w", "t2f")

def discover_cases(data_root):
    root = Path(data_root)
    candidates = list(root.rglob("*-t1n.nii")) + list(root.rglob("*-t1n.nii.gz"))
    cases = {}

    for t1n_path in candidates:
        name = t1n_path.name
        suffix = "-t1n.nii.gz" if name.endswith("-t1n.nii.gz") else "-t1n.nii"
        case_id = name[:-len(suffix)]
        case_dir = t1n_path.parent

        files = {}
        for modality in MODALITIES:
            paths = [
                case_dir / f"{case_id}-{modality}.nii",
                case_dir / f"{case_id}-{modality}.nii.gz"
            ]
            matches = [p for p in paths if p.exists()]
            if len(matches) != 1:
                raise FileNotFoundError(
                    f"{case_id}: expected one {modality} file, found {matches}"
                )
            files[modality] = matches[0]

        seg_paths = [
            case_dir / f"{case_id}-seg.nii",
            case_dir / f"{case_id}-seg.nii.gz"
        ]
        seg_matches = [p for p in seg_paths if p.exists()]
        if len(seg_matches) != 1:
            raise FileNotFoundError(
                f"{case_id}: expected one segmentation file, found {seg_matches}"
            )

        files["seg"] = seg_matches[0]
        cases[case_id] = files

    if not cases:
        raise FileNotFoundError(
            f"No '*-t1n.nii[.gz]' files found under {root}. Check DATA_ROOT."
        )

    return dict(sorted(cases.items()))

case_files = discover_cases(DATA_ROOT)
case_ids = list(case_files.keys())

print("Number of discovered cases:", len(case_ids))
print("First five cases:", case_ids[:5])

## 3. Load and validate one patient

NIfTI files store both voxel data and spatial metadata (affine). We check:
- all four modalities have the same shape;
- all four modalities have matching affines;
- the segmentation matches the modalities;
- labels belong to the BraTS 2024 set `{0, 1, 2, 3, 4}`.

We use canonical orientation for all files. This changes axis ordering when needed, but does not resample voxel values.

In [ ]:
def load_case(case_files, canonical=True):
    image_arrays = []
    reference_shape = None
    reference_affine = None

    for modality in MODALITIES:
        nii = nib.load(str(case_files[modality]))
        if canonical:
            nii = nib.as_closest_canonical(nii)

        array = np.asarray(nii.dataobj, dtype=np.float32)

        if reference_shape is None:
            reference_shape = array.shape
            reference_affine = nii.affine
        else:
            if array.shape != reference_shape:
                raise ValueError(
                    f"{modality} shape mismatch: {array.shape} vs {reference_shape}"
                )
            if not np.allclose(nii.affine, reference_affine, atol=1e-4):
                raise ValueError(f"{modality} affine does not match reference")

        image_arrays.append(array)

    seg_nii = nib.load(str(case_files["seg"]))
    if canonical:
        seg_nii = nib.as_closest_canonical(seg_nii)

    label = np.asarray(seg_nii.dataobj, dtype=np.float32)

    if label.shape != reference_shape:
        raise ValueError(
            f"Segmentation shape mismatch: {label.shape} vs {reference_shape}"
        )
    if not np.allclose(seg_nii.affine, reference_affine, atol=1e-4):
        raise ValueError("Segmentation affine does not match MRI modalities")

    label = np.rint(label).astype(np.int16)

    allowed_labels = {0, 1, 2, 3, 4}
    observed_labels = set(np.unique(label).tolist())
    if not observed_labels.issubset(allowed_labels):
        raise ValueError(f"Unexpected labels: {observed_labels - allowed_labels}")

    image = np.stack(image_arrays, axis=0).astype(np.float32)
    return image, label

example_id = case_ids[0]
example_image, example_label = load_case(case_files[example_id])

print("Patient:", example_id)
print("Image shape [C, X, Y, Z]:", example_image.shape)
print("Segmentation shape [X, Y, Z]:", example_label.shape)
print("Unique labels:", np.unique(example_label))
print("Voxel spacing:", nib.load(str(case_files["t1n"])).header.get_zooms()[:3])

## 4. Inspect modality intensity distributions

MRI intensities are not on a universal physical scale like CT Hounsfield units. We therefore inspect each modality before selecting a normalization strategy.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3))

for channel, (ax, modality) in enumerate(zip(axes, MODALITIES)):
    volume = example_image[channel]
    nonzero = volume[np.abs(volume) > 1e-6]

    ax.hist(nonzero[::max(1, len(nonzero)//50000)], bins=80)
    ax.set_title(modality.upper())
    ax.set_xlabel("Raw intensity")
    ax.set_ylabel("Voxel count")

plt.suptitle(f"Raw intensity distributions — {example_id}")
plt.tight_layout()
plt.show()

## 5. Intensity normalization

We use **per-patient, per-modality z-score normalization** over non-zero voxels:

\[
x' = \frac{x-\mu}{\sigma+\epsilon}
\]

where the mean and standard deviation are calculated separately for each MRI modality and patient. Zero-valued background voxels remain zero.

This is a practical baseline for the custom models, not a claim that it is universally optimal. We will document this choice and keep it consistent across the custom-model experiments.

In [ ]:
def normalize_modalities(image, eps=1e-8):
    image = np.asarray(image, dtype=np.float32)
    normalized = np.zeros_like(image, dtype=np.float32)

    for channel in range(image.shape[0]):
        volume = image[channel]
        valid = np.isfinite(volume) & (np.abs(volume) > eps)

        if not np.any(valid):
            continue

        values = volume[valid]
        mean = values.mean(dtype=np.float64)
        std = values.std(dtype=np.float64)

        if std < eps:
            std = 1.0

        normalized[channel, valid] = (volume[valid] - mean) / std

    return np.nan_to_num(
        normalized, copy=False, nan=0.0, posinf=0.0, neginf=0.0
    )

normalized_example = normalize_modalities(example_image)

for channel, modality in enumerate(MODALITIES):
    values = normalized_example[channel][
        np.abs(example_image[channel]) > 1e-6
    ]
    print(
        f"{modality.upper():4s} | mean={values.mean(): .4f} | "
        f"std={values.std():.4f}"
    )

## 6. Preserve the BraTS 2024 segmentation labels

We do not remap the labels. The target remains an integer-valued volume:

| Value | Region |
|---:|---|
| 0 | Background |
| 1 | NETC |
| 2 | SNFH |
| 3 | ET |
| 4 | RC |

The model will later produce five output channels (one per class). For `CrossEntropyLoss`, targets must be integer class IDs with shape `[B, D, H, W]`, not one-hot encoded.

In [ ]:
label_values, label_counts = np.unique(example_label, return_counts=True)

label_summary = pd.DataFrame({
    "label": label_values,
    "voxel_count": label_counts,
    "percentage": 100 * label_counts / example_label.size
})

display(label_summary)

## 7. Patient-level train/validation/test split

We split by **patient**, never by slices or patches. This prevents patches from the same patient appearing in both training and validation/test sets.

We use a reproducible 70/15/15 split and save it to disk so every custom model uses exactly the same patient partitions.

In [ ]:
from sklearn.model_selection import train_test_split

train_ids, remaining_ids = train_test_split(
    case_ids,
    test_size=0.30,
    random_state=SEED,
    shuffle=True
)

val_ids, test_ids = train_test_split(
    remaining_ids,
    test_size=0.50,
    random_state=SEED,
    shuffle=True
)

# Explicitly verify that no patient appears in more than one partition.
assert set(train_ids).isdisjoint(val_ids)
assert set(train_ids).isdisjoint(test_ids)
assert set(val_ids).isdisjoint(test_ids)
assert set(train_ids) | set(val_ids) | set(test_ids) == set(case_ids)

split_df = pd.DataFrame(
    [(case_id, "train") for case_id in train_ids] +
    [(case_id, "validation") for case_id in val_ids] +
    [(case_id, "test") for case_id in test_ids],
    columns=["case_id", "split"]
).sort_values("case_id").reset_index(drop=True)

split_path = WORK_DIR / "patient_splits.csv"
split_df.to_csv(split_path, index=False)

print(split_df["split"].value_counts())
print("Saved split manifest:", split_path)

## 8. 3D patch extraction

Training on an entire 3D MRI volume is memory-intensive. We therefore train on smaller 3D patches.

For the initial pipeline:
- Patch size: `64 × 64 × 64`.
- Patch dimensions are divisible by 16, matching the four downsampling stages in our current 3D U-Net.
- With probability 0.7, the patch centre is sampled from a non-background voxel.
- Otherwise, the centre is sampled randomly from the volume.

This tumour-aware strategy reduces the chance of repeatedly sampling only background. It does **not** guarantee that every tumour subregion (especially small classes) appears in every batch. We will inspect class frequencies in sampled patches and adjust if necessary.

In [ ]:
def crop_with_padding(array, center, patch_size):
    spatial_shape = array.shape[-3:]
    starts = [int(center[i]) - patch_size[i] // 2 for i in range(3)]
    ends = [starts[i] + patch_size[i] for i in range(3)]

    src_starts = [max(0, starts[i]) for i in range(3)]
    src_ends = [min(spatial_shape[i], ends[i]) for i in range(3)]
    pad_before = [max(0, -starts[i]) for i in range(3)]
    pad_after = [max(0, ends[i] - spatial_shape[i]) for i in range(3)]

    slices = tuple(slice(src_starts[i], src_ends[i]) for i in range(3))

    if array.ndim == 4:
        cropped = array[(slice(None),) + slices]
        pad_width = [(0, 0)] + list(zip(pad_before, pad_after))
    else:
        cropped = array[slices]
        pad_width = list(zip(pad_before, pad_after))

    return np.pad(cropped, pad_width, mode="constant", constant_values=0)


def extract_patch(
    image,
    label,
    patch_size=(64, 64, 64),
    positive_probability=0.7,
    rng=None
):
    if rng is None:
        rng = np.random.default_rng()

    patch_size = tuple(int(v) for v in patch_size)
    if len(patch_size) != 3 or any(v <= 0 for v in patch_size):
        raise ValueError("patch_size must contain three positive integers")

    positive_coords = np.argwhere(label > 0)

    if len(positive_coords) > 0 and rng.random() < positive_probability:
        center = positive_coords[rng.integers(0, len(positive_coords))]
    else:
        center = np.array([
            rng.integers(0, label.shape[axis]) for axis in range(3)
        ])

    image_patch = crop_with_padding(image, center, patch_size)
    label_patch = crop_with_padding(label, center, patch_size)

    return (
        image_patch.astype(np.float32),
        label_patch.astype(np.int64)
    )


PATCH_SIZE = (64, 64, 64)

patch_image, patch_label = extract_patch(
    normalized_example,
    example_label,
    patch_size=PATCH_SIZE,
    positive_probability=0.7,
    rng=np.random.default_rng(SEED)
)

print("Patch image shape:", patch_image.shape)
print("Patch label shape:", patch_label.shape)
print("Patch labels:", np.unique(patch_label))

assert patch_image.shape == (4, *PATCH_SIZE)
assert patch_label.shape == PATCH_SIZE

## 9. Create a reusable PyTorch Dataset module

The module below is saved to the working directory. Later custom-model notebooks can import the same loading, normalization, patch extraction and Dataset code rather than duplicating it.

The Dataset loads cases lazily instead of keeping all 700 full 3D volumes in RAM. `patches_per_case` controls how many patches each patient contributes per epoch.

In [ ]:
MODULE_DIR = Path("/kaggle/working/src/data")
MODULE_DIR.mkdir(parents=True, exist_ok=True)

module_source = "from pathlib import Path\nfrom collections import OrderedDict\nimport numpy as np\nimport nibabel as nib\nimport torch\nfrom torch.utils.data import Dataset\n\nMODALITIES = (\"t1n\", \"t1c\", \"t2w\", \"t2f\")\nLABEL_NAME_TO_ID = {\"NETC\": 1, \"SNFH\": 2, \"ET\": 3, \"RC\": 4}\n\ndef discover_cases(data_root):\n    \"\"\"Find cases by locating each *-t1n.nii or *-t1n.nii.gz file.\"\"\"\n    root = Path(data_root)\n    candidates = list(root.rglob(\"*-t1n.nii\")) + list(root.rglob(\"*-t1n.nii.gz\"))\n    cases = {}\n    for t1n_path in candidates:\n        name = t1n_path.name\n        suffix = \"-t1n.nii.gz\" if name.endswith(\"-t1n.nii.gz\") else \"-t1n.nii\"\n        case_id = name[:-len(suffix)]\n        case_dir = t1n_path.parent\n        files = {}\n        for modality in MODALITIES:\n            expected = [\n                case_dir / f\"{case_id}-{modality}.nii\",\n                case_dir / f\"{case_id}-{modality}.nii.gz\",\n            ]\n            matches = [p for p in expected if p.exists()]\n            if len(matches) != 1:\n                raise FileNotFoundError(\n                    f\"Expected exactly one file for {case_id}, modality {modality}; found {matches}\"\n                )\n            files[modality] = matches[0]\n        seg_candidates = [\n            case_dir / f\"{case_id}-seg.nii\",\n            case_dir / f\"{case_id}-seg.nii.gz\",\n        ]\n        seg_matches = [p for p in seg_candidates if p.exists()]\n        if len(seg_matches) != 1:\n            raise FileNotFoundError(\n                f\"Expected exactly one segmentation for {case_id}; found {seg_matches}\"\n            )\n        files[\"seg\"] = seg_matches[0]\n        cases[case_id] = files\n    if not cases:\n        raise FileNotFoundError(\n            f\"No '*-t1n.nii[.gz]' files found under {root}. Check DATA_ROOT.\"\n        )\n    return dict(sorted(cases.items()))\n\ndef load_case(case_files, canonical=True):\n    \"\"\"\n    Load four MRI modalities and the segmentation.\n    Returns image [4, X, Y, Z] float32 and label [X, Y, Z] int16.\n    No resampling is performed; spatial alignment is checked.\n    \"\"\"\n    arrays, affines = [], []\n    reference_shape = None\n    reference_affine = None\n\n    for modality in MODALITIES:\n        nii = nib.load(str(case_files[modality]))\n        if canonical:\n            nii = nib.as_closest_canonical(nii)\n        arr = np.asarray(nii.dataobj, dtype=np.float32)\n\n        if reference_shape is None:\n            reference_shape = arr.shape\n            reference_affine = nii.affine\n        else:\n            if arr.shape != reference_shape:\n                raise ValueError(\n                    f\"Shape mismatch for {modality}: {arr.shape} vs {reference_shape}\"\n                )\n            if not np.allclose(nii.affine, reference_affine, atol=1e-4):\n                raise ValueError(f\"Affine mismatch for modality {modality}\")\n\n        arrays.append(arr)\n\n    seg_nii = nib.load(str(case_files[\"seg\"]))\n    if canonical:\n        seg_nii = nib.as_closest_canonical(seg_nii)\n    label = np.asarray(seg_nii.dataobj, dtype=np.float32)\n\n    if label.shape != reference_shape:\n        raise ValueError(f\"Segmentation shape mismatch: {label.shape} vs {reference_shape}\")\n    if not np.allclose(seg_nii.affine, reference_affine, atol=1e-4):\n        raise ValueError(\"Segmentation affine does not match MRI modalities\")\n\n    # Labels are categorical integers, not continuous intensities.\n    label = np.rint(label).astype(np.int16)\n    observed = set(np.unique(label).tolist())\n    allowed = {0, 1, 2, 3, 4}\n    if not observed.issubset(allowed):\n        raise ValueError(f\"Unexpected segmentation labels: {sorted(observed - allowed)}\")\n\n    image = np.stack(arrays, axis=0).astype(np.float32)\n    return image, label\n\ndef normalize_modalities(image, eps=1e-8):\n    \"\"\"\n    Per-volume, per-modality z-score normalization over non-zero voxels.\n    Background remains zero. Input/output shape: [4, X, Y, Z].\n    \"\"\"\n    image = np.asarray(image, dtype=np.float32)\n    normalized = np.zeros_like(image, dtype=np.float32)\n\n    for channel in range(image.shape[0]):\n        volume = image[channel]\n        valid = np.isfinite(volume) & (np.abs(volume) > eps)\n        if not np.any(valid):\n            continue\n\n        values = volume[valid]\n        mean = values.mean(dtype=np.float64)\n        std = values.std(dtype=np.float64)\n        if std < eps:\n            std = 1.0\n\n        normalized_channel = np.zeros_like(volume, dtype=np.float32)\n        normalized_channel[valid] = (volume[valid] - mean) / std\n        normalized[channel] = normalized_channel\n\n    # Replace any numerical issues defensively.\n    np.nan_to_num(normalized, copy=False, nan=0.0, posinf=0.0, neginf=0.0)\n    return normalized\n\ndef _crop_with_padding(array, center, patch_size):\n    \"\"\"Crop a [C,X,Y,Z] or [X,Y,Z] array around center; pad if needed.\"\"\"\n    spatial_shape = array.shape[-3:]\n    starts = [int(center[i]) - patch_size[i] // 2 for i in range(3)]\n    ends = [starts[i] + patch_size[i] for i in range(3)]\n\n    src_starts = [max(0, starts[i]) for i in range(3)]\n    src_ends = [min(spatial_shape[i], ends[i]) for i in range(3)]\n    pad_before = [max(0, -starts[i]) for i in range(3)]\n    pad_after = [max(0, ends[i] - spatial_shape[i]) for i in range(3)]\n\n    slices = tuple(slice(src_starts[i], src_ends[i]) for i in range(3))\n    if array.ndim == 4:\n        cropped = array[(slice(None),) + slices]\n        pad_width = [(0, 0)] + list(zip(pad_before, pad_after))\n    else:\n        cropped = array[slices]\n        pad_width = list(zip(pad_before, pad_after))\n\n    return np.pad(cropped, pad_width, mode=\"constant\", constant_values=0)\n\ndef extract_patch(image, label, patch_size=(64, 64, 64),\n                  positive_probability=0.7, rng=None):\n    \"\"\"\n    Extract a patch from image [C,X,Y,Z] and label [X,Y,Z].\n    With probability positive_probability, center on a non-background voxel.\n    Otherwise choose a random spatial center.\n    \"\"\"\n    if rng is None:\n        rng = np.random.default_rng()\n\n    patch_size = tuple(int(v) for v in patch_size)\n    if len(patch_size) != 3 or any(v <= 0 for v in patch_size):\n        raise ValueError(\"patch_size must contain three positive integers\")\n\n    spatial_shape = label.shape\n    positive_coords = np.argwhere(label > 0)\n\n    choose_positive = len(positive_coords) > 0 and rng.random() < positive_probability\n    if choose_positive:\n        center = positive_coords[rng.integers(0, len(positive_coords))]\n    else:\n        center = np.array([\n            rng.integers(0, spatial_shape[axis]) for axis in range(3)\n        ])\n\n    image_patch = _crop_with_padding(image, center, patch_size)\n    label_patch = _crop_with_padding(label, center, patch_size)\n\n    return image_patch.astype(np.float32), label_patch.astype(np.int64)\n\nclass BraTS3DPatchDataset(Dataset):\n    \"\"\"\n    Lazy-loading patch dataset.\n    One index corresponds to one sampled patch; cases can be repeated\n    using patches_per_case. Each item returns:\n      image: float32 [4, D, H, W]\n      label: int64   [D, H, W]\n    \"\"\"\n    def __init__(self, case_ids, case_files, patch_size=(64, 64, 64),\n                 patches_per_case=1, positive_probability=0.7,\n                 training=True, seed=42, canonical=True):\n        self.case_ids = list(case_ids)\n        self.case_files = case_files\n        self.patch_size = tuple(patch_size)\n        self.patches_per_case = int(patches_per_case)\n        self.positive_probability = float(positive_probability)\n        self.training = bool(training)\n        self.seed = int(seed)\n        self.canonical = bool(canonical)\n\n        if not self.case_ids:\n            raise ValueError(\"case_ids cannot be empty\")\n        if self.patches_per_case < 1:\n            raise ValueError(\"patches_per_case must be >= 1\")\n\n    def __len__(self):\n        return len(self.case_ids) * self.patches_per_case\n\n    def __getitem__(self, index):\n        case_index = index % len(self.case_ids)\n        case_id = self.case_ids[case_index]\n\n        image, label = load_case(self.case_files[case_id], canonical=self.canonical)\n        image = normalize_modalities(image)\n\n        # Deterministic sampling for inspection when training=False.\n        if self.training:\n            rng = np.random.default_rng()\n        else:\n            rng = np.random.default_rng(self.seed + index)\n\n        image_patch, label_patch = extract_patch(\n            image,\n            label,\n            patch_size=self.patch_size,\n            positive_probability=self.positive_probability,\n            rng=rng,\n        )\n\n        image_tensor = torch.from_numpy(np.ascontiguousarray(image_patch))\n        label_tensor = torch.from_numpy(np.ascontiguousarray(label_patch)).long()\n\n        return image_tensor, label_tensor\n"
module_path = MODULE_DIR / "neuroseg_data.py"
module_path.write_text(module_source)

print("Reusable module saved to:", module_path)

## 10. Build and inspect the PyTorch DataLoader

We use a small batch size for the first smoke test because 3D activations consume substantial GPU memory.

- Image batch: `[B, 4, D, H, W]`, `float32`
- Label batch: `[B, D, H, W]`, `int64`

The training Dataset samples random patches. For validation/test, this patch Dataset is only useful for pipeline inspection; final evaluation should use full-volume or sliding-window inference so the reported metrics represent complete patients.

In [ ]:
# Import the reusable module we just wrote.
sys.path.insert(0, "/kaggle/working/src/data")

from neuroseg_data import (
    discover_cases as discover_cases_module,
    BraTS3DPatchDataset
)

# Reuse the patient split saved above.
train_dataset = BraTS3DPatchDataset(
    case_ids=train_ids,
    case_files=case_files,
    patch_size=PATCH_SIZE,
    patches_per_case=1,
    positive_probability=0.7,
    training=True,
    seed=SEED
)

train_loader = DataLoader(
    train_dataset,
    batch_size=1,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

images, labels = next(iter(train_loader))

print("Image batch:", images.shape, images.dtype)
print("Label batch:", labels.shape, labels.dtype)
print("Unique labels in this batch:", torch.unique(labels))

## 11. Visualize one sampled patch

The display below shows one central slice from each MRI modality and the corresponding segmentation mask. The first tensor axis is treated as the slice axis for this visualization; it is a visualization convention, not a claim about anatomical orientation.

In [ ]:
sample_image = images[0].numpy()  # [4, D, H, W]
sample_label = labels[0].numpy()  # [D, H, W]

slice_idx = sample_image.shape[1] // 2

fig, axes = plt.subplots(1, 5, figsize=(18, 4))

for channel, modality in enumerate(MODALITIES):
    axes[channel].imshow(sample_image[channel, slice_idx], cmap="gray")
    axes[channel].set_title(modality.upper())
    axes[channel].axis("off")

axes[4].imshow(sample_label[slice_idx], cmap="tab10", vmin=0, vmax=4)
axes[4].set_title("Segmentation")
axes[4].axis("off")

plt.suptitle("One sampled 3D patch — central slice")
plt.tight_layout()
plt.show()

## 12. Inspect class distribution in sampled patches

Tumour-aware sampling increases the chance of seeing tumour voxels, but class imbalance can remain. We record the class composition of this sample as a sanity check. Later, training loss and evaluation will account for the imbalance more carefully.

In [ ]:
patch_values, patch_counts = np.unique(sample_label, return_counts=True)

patch_distribution = pd.DataFrame({
    "label": patch_values,
    "voxel_count": patch_counts,
    "percentage": 100 * patch_counts / sample_label.size
})

display(patch_distribution)

## 13. Save metadata and reproducibility information

We save the split manifest and a small JSON configuration. The raw MRI data remains in Kaggle's read-only input directory; we do not duplicate the complete dataset.

In [ ]:
config = {
    "dataset": "BraTS 2024 Adult Glioma",
    "number_of_cases": len(case_ids),
    "split_seed": SEED,
    "split_ratios": {"train": 0.70, "validation": 0.15, "test": 0.15},
    "modalities": list(MODALITIES),
    "labels": {
        "0": "Background",
        "1": "NETC",
        "2": "SNFH",
        "3": "ET",
        "4": "RC"
    },
    "normalization": "Per-patient, per-modality z-score over non-zero voxels",
    "patch_size": list(PATCH_SIZE),
    "positive_patch_probability": 0.7,
    "patch_sampling": "Tumour-centred with random-centre fallback",
    "notes": [
        "Patient-level split; no patient overlap across partitions.",
        "No resampling is performed in this custom pipeline.",
        "nnU-Net should use its own self-configuring preprocessing pipeline.",
        "Final validation/test evaluation should use full-volume or sliding-window inference."
    ]
}

config_path = WORK_DIR / "preprocessing_config.json"
config_path.write_text(json.dumps(config, indent=4))

print("Saved:", split_path)
print("Saved:", config_path)
print("Saved:", module_path)

## 14. Completion checklist and next step

Before moving to model training, confirm:

- [ ] All expected patient cases were discovered.
- [ ] Four MRI modalities and segmentation have matching shapes and affines.
- [ ] Only labels 0–4 are present.
- [ ] Normalized non-zero voxels have approximately zero mean and unit standard deviation.
- [ ] Train/validation/test patient IDs do not overlap.
- [ ] Patch tensors have the expected dimensions and dtypes.
- [ ] The sampled patch visualization looks spatially aligned.

### Important limitations to carry into training
1. The 70/15/15 split is reproducible but not necessarily stratified by tumour subtype or region prevalence. We should inspect split-level label statistics before final experiments.
2. Per-volume normalization is a baseline choice. Do not change it for one model only if we intend to compare models fairly.
3. Tumour-centred sampling is not a substitute for class-aware sampling. Rare labels may still be underrepresented.
4. `patches_per_case=1` is only a smoke-test setting. Training can use more patches per patient, after checking runtime and memory.
5. nnU-Net uses its own preprocessing and patch sampling logic.

**Next notebook:** Notebook 07 — 3D U-Net architecture, training loop, checkpointing, validation and full-volume/sliding-window evaluation.